# ENVIRONMENT

In [1]:
from gymnasium import Env 
from gymnasium.spaces import MultiBinary, Box
import stable_retro as retro
import numpy as np
import cv2
from matplotlib import pyplot as plt
import time
import optuna
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
import os
from collections import deque

/home/master26/miniconda3/envs/sfrl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# record = '/home/master26/Documents/sfrl/replays'
record = False
# render_mode = "human"
render_mode = False
game = 'StreetFighterIISpecialChampionEdition-Genesis-v0'
state = "//home/master26/Documents/sfrl/FightLadder/data/sf/curriculum/Level1.20.state"
frame_rate = 1/115

In [3]:
# Create custom environment 
class StreetFighter(Env): 
    def __init__(self, record=record, render_mode=render_mode):
        super().__init__()
        # Specify action space and observation space 
        self.observation_space = Box(low=0, high=255, shape=(84, 84, 1), dtype=np.uint8)
        self.action_space = MultiBinary(12)
        # Startup and instance of the game 
        self.game = retro.make(game=game, 
                               state=state,
                               record=record,
                               render_mode=render_mode,
                               use_restricted_actions=retro.Actions.FILTERED)
    
    def reset(self, **kwargs):
        # Return the first frame 
        obs, info = self.game.reset()
        obs = self.preprocess(obs) 
        self.previous_frame = obs 
        
        # Create a attribute to hold the score delta 
        self.health = 176
        self.enemy_health = 176
        return obs, info
    
    def preprocess(self, observation): 
        # Grayscaling 
        gray = cv2.cvtColor(observation, cv2.COLOR_BGR2GRAY)
        # Resize 
        resize = cv2.resize(gray, (84,84), interpolation=cv2.INTER_CUBIC)
        # Add the channels value
        channels = np.reshape(resize, (84,84,1))
        return channels 
    
    def step(self, action): 
        # Take a step 
        obs, reward, done, truncated, info = self.game.step(action)
        obs = self.preprocess(obs) 
        
        # Frame delta 
        frame_delta = obs - self.previous_frame
        self.previous_frame = obs 
        
        # Reshape the reward function
        reward = (info["health"] - self.health) + (self.enemy_health - info["enemy_health"])
        self.health = info["health"]
        self.enemy_health = info["enemy_health"]
        
        return frame_delta, reward, done, truncated, info
    
    def render(self, *args, **kwargs):
        self.game.render()
        
    def close(self):
        self.game.close()
        if render_mode:
            self.game.viewer.close()

In [ ]:
# Random agent
env = StreetFighter(render_mode="human", record=False)
# Reset game to starting state
obs, _ = env.reset()
wins = 0
rounds_wins = 0
total_rounds = 0
# Set flag to flase
done = False
episodes = 1
infos = []
for game in range(episodes): 
    while not done: 
        obs, reward, done, truncated, info = env.step(env.action_space.sample())
        # if reward != 0:
        #     print(reward)
        time.sleep(frame_rate)
        if info["enemy_health"] < 0:
            rounds_wins += 1
        if (info["enemy_health"] < 0) or (info["health"] < 0):
            total_rounds += 1

    infos.append(info)
        
    if info["matches_won"]==2:
        print("MATCH WON")
        wins += 1
    else:
        print("MATCH LOST")
    print(f"Player: {info['matches_won']} - CPU: {info['enemy_matches_won']}")
    obs, _ = env.reset()

# env.close()

# print("winrate = %.2f"%(wins/episodes))
# print("rounds winrate = %.2f"%(rounds_wins/total_rounds))

In [11]:
info

{'enemy_y_position': 172,
 'enemy_character': 4,
 'enemy_matches_won': 2,
 'enemy_status': 522,
 'enemy_x_position': 161,
 'matches_won': 0,
 'continue_timer': 9,
 'status': 1024,
 'y_position': 192,
 'health': -1,
 'round_timer': 33042,
 'x_position': 129,
 'score': 312001,
 'enemy_health': 147}

# Agent Optimization

In [5]:
LOG_DIR = './logs/'
OPT_DIR = './opt/'

In [6]:
SAVE_PATH = os.path.join(OPT_DIR, 'trial_{}_best_model'.format(1))

In [7]:
# Function to return test hyperparameters - define the object function
def optimize_ppo(trial): 
    return {
        'n_steps':trial.suggest_int('n_steps', 2048, 8192),
        'gamma':trial.suggest_loguniform('gamma', 0.8, 0.9999),
        'learning_rate':trial.suggest_loguniform('learning_rate', 1e-5, 1e-4),
        'clip_range':trial.suggest_uniform('clip_range', 0.1, 0.4),
        'gae_lambda':trial.suggest_uniform('gae_lambda', 0.8, 0.99)
    }

In [8]:
# Run a training loop and return mean reward 
def optimize_agent(trial):
    # try:
    model_params = optimize_ppo(trial) 

    # Create environment 
    env = StreetFighter()
    env = Monitor(env, LOG_DIR, info_keywords=("matches_won", "enemy_matches_won", "health", "enemy_health"))
    env = DummyVecEnv([lambda: env])
    env = VecFrameStack(env, 4, channels_order='last')

    # Create algo 
    model = PPO('CnnPolicy', env, tensorboard_log=LOG_DIR, verbose=0, **model_params)
    model.learn(total_timesteps=100000)

    # Evaluate model 
    mean_reward, _ = evaluate_policy(model, env, n_eval_episodes=5)
    env.close()

    SAVE_PATH = os.path.join(OPT_DIR, 'trial_{}_best_model'.format(trial.number))
    model.save(SAVE_PATH)

    return mean_reward

    # except Exception as e:
    #     return -1000

In [9]:
# Creating the experiment 
study = optuna.create_study(direction='maximize')
study.optimize(optimize_agent, n_trials=100, n_jobs=1)


[I 2026-02-19 23:59:45,640] A new study created in memory with name: no-name-2b86c746-89eb-4ddb-9416-881554282dec
/tmp/ipykernel_14531/3802630500.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'gamma':trial.suggest_loguniform('gamma', 0.8, 0.9999),
/tmp/ipykernel_14531/3802630500.py:6: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  'learning_rate':trial.suggest_loguniform('learning_rate', 1e-5, 1e-4),
/tmp/ipykernel_14531/3802630500.py:7: FutureWarning: suggest_uniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float instead.
  'clip_range':trial.suggest_uniform('clip_r

In [12]:
study.best_params

{'n_steps': 7195,
 'gamma': 0.9932785893338508,
 'learning_rate': 1.4593792748053036e-05,
 'clip_range': 0.11424933185397193,
 'gae_lambda': 0.822954020880725}

In [13]:
study.best_trial

FrozenTrial(number=0, state=<TrialState.COMPLETE: 1>, values=[335801.0], datetime_start=datetime.datetime(2026, 2, 19, 23, 59, 45, 642225), datetime_complete=datetime.datetime(2026, 2, 20, 0, 2, 56, 193034), params={'n_steps': 7195, 'gamma': 0.9932785893338508, 'learning_rate': 1.4593792748053036e-05, 'clip_range': 0.11424933185397193, 'gae_lambda': 0.822954020880725}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_steps': IntDistribution(high=8192, log=False, low=2048, step=1), 'gamma': FloatDistribution(high=0.9999, log=True, low=0.8, step=None), 'learning_rate': FloatDistribution(high=0.0001, log=True, low=1e-05, step=None), 'clip_range': FloatDistribution(high=0.4, log=False, low=0.1, step=None), 'gae_lambda': FloatDistribution(high=0.99, log=False, low=0.8, step=None)}, trial_id=0, value=None)

# Training Best Agent

In [10]:
num_timesteps = 1_000_000

In [11]:
# Import base callback 
from stable_baselines3.common.callbacks import BaseCallback

In [12]:
class TrainAndLoggingCallback(BaseCallback):

    def __init__(self, check_freq, save_path, verbose=1):
        super(TrainAndLoggingCallback, self).__init__(verbose)
        self.check_freq = check_freq
        self.save_path = save_path
        self.last_model_path = None  # track previous checkpoint
        self.total_timesteps = num_timesteps

    def _init_callback(self):
        if self.save_path is not None:
            os.makedirs(self.save_path, exist_ok=True)
        self.start_time = time.time()  # Record start time


    def _on_step(self):
        if self.n_calls % self.check_freq == 0:

            # Calculate progress and elapsed time
            progress = (self.n_calls / self.total_timesteps) * 100
            elapsed_time = (time.time() - self.start_time) / 60  # Convert to minutes
            
            # Print progress info
            print(f"n_calls: {self.n_calls}/{self.total_timesteps} ({progress:.2f}%) - "
                  f"Elapsed time: {elapsed_time: .1f} minutes")
            
            # save new checkpoint
            model_path = os.path.join(
                self.save_path,
                f'best_model_{self.n_calls}'
            )
            self.model.save(model_path)

            # delete previous checkpoint if it exists
            if self.last_model_path is not None and os.path.exists(self.last_model_path):
                os.remove(self.last_model_path)


            # update pointer
            self.last_model_path = model_path


        return True

In [13]:
CHECKPOINT_DIR = './train/'

In [14]:
callback = TrainAndLoggingCallback(check_freq=num_timesteps/100, save_path=CHECKPOINT_DIR)

In [15]:
# Create environment 
env = StreetFighter()
env = Monitor(env, LOG_DIR, info_keywords=("matches_won", "enemy_matches_won", "health", "enemy_health"))
env = DummyVecEnv([lambda: env])
env = VecFrameStack(env, 4, channels_order='last')

In [16]:
model_params = study.best_params
model_params['n_steps'] = 7488  # set n_steps to 7488 or a factor of 64
# model_params['learning_rate'] = 5e-7
model_params

NameError: name 'study' is not defined

In [19]:
# model = PPO('CnnPolicy', env, tensorboard_log=LOG_DIR, verbose=1, **model_params)

model = PPO('CnnPolicy', env, tensorboard_log=LOG_DIR, verbose=1)

Using cuda device
Wrapping the env in a VecTransposeImage.


In [ ]:
# # Reload previous weights from HPO
# model.load(os.path.join(OPT_DIR, 'trial_0_best_model.zip'))

In [ ]:
# Kick off training 
model.learn(total_timesteps=num_timesteps, callback=callback)
# model.learn(total_timestep=5000000) 

Logging to ./logs/PPO_1
-----------------------------
| time/              |      |
|    fps             | 650  |
|    iterations      | 1    |
|    time_elapsed    | 3    |
|    total_timesteps | 2048 |
-----------------------------
---------------------------------------
| rollout/                |           |
|    ep_len_mean          | 3.41e+03  |
|    ep_rew_mean          | -94       |
| time/                   |           |
|    fps                  | 648       |
|    iterations           | 2         |
|    time_elapsed         | 6         |
|    total_timesteps      | 4096      |
| train/                  |           |
|    approx_kl            | 2.0040846 |
|    clip_fraction        | 0.605     |
|    clip_range           | 0.2       |
|    entropy_loss         | -7.61     |
|    explained_variance   | 0.000235  |
|    learning_rate        | 0.0003    |
|    loss                 | 16.6      |
|    n_updates            | 10        |
|    policy_gradient_loss | 0.0817    |
|    v

# EVALUATION

In [9]:
model = PPO.load('./train/best_model_15000000.zip')

In [10]:
mean_reward, _ = evaluate_policy(model, env, render=True, n_eval_episodes=1)

NameError: name 'env' is not defined

In [11]:
render_mode="human"
record = '/home/master26/Documents/sfrl/replays'
env = StreetFighter(render_mode=render_mode, record=record)
# Reset game to starting state
obs = env.reset()
wins = 0
rounds_wins = 0
total_rounds = 0
# Set flag to flase
done = False
episodes = 1

frames = deque(maxlen=4)
for game in range(episodes): 
    while not done: 

        if len(frames) < 4:
            action = np.zeros(12, dtype=np.int8)
        else:
            action = model.predict(np.squeeze(np.stack(frames)))[0]

        obs, reward, done, truncated, info = env.step(action)
        time.sleep(frame_rate)

        frames.append(obs)
        if info["enemy_health"] < 0:
            print("ROUND WON")
            rounds_wins += 1
        if (info["enemy_health"] < 0) or (info["health"] < 0):
            total_rounds += 1

        
    if info["matches_won"]==2:
        print("MATCH WON")
        wins += 1
    else:
        print("MATCH LOST")
    print(f"Player: {info['matches_won']} - CPU: {info['enemy_matches_won']}")
    obs = env.reset()

env.close()

MATCH LOST
Player: 0 - CPU: 2


In [ ]:
# Reset game to starting state
obs = env.reset()
# Set flag to flase
done = False
for game in range(1): 
    while not done: 
        if done: 
            obs = env.reset()
        env.render()
        action = model.predict(obs)[0]
        obs, reward, done, info = env.step(action)
        time.sleep(frame_rate)
